<a href="https://colab.research.google.com/github/fianbio/AI-driven-PLA2-antivenome/blob/main/03a_model_ensemble_qsar_morgan.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Ensemble QSAR Pipeline: Scaffold Split + Optuna Tuning + Stacking

Pipeline untuk data **Morgan FP + PubChem FP + RDKit Descriptors** (classification).

Ensemble yang dibangun: **RF+XGB**, **RF+LGB**, **RF+CatBoost** (via Stacking).

Validasi menggunakan **scaffold-based GroupKFold** untuk mencegah data leakage antar molekul mirip.


## 1. Instalasi Library (khusus Colab)

In [ ]:
!pip install -q rdkit xgboost lightgbm catboost optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.1/38.1 MB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 442.4/442.4 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.7/268.7 kB 4.9 MB/s eta 0:00:00


## 2. Import Library

In [ ]:
import numpy as np
import pandas as pd
import optuna
from sklearn.model_selection import GroupKFold
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

from rdkit import Chem
from rdkit.Chem.Scaffolds import MurckoScaffold

optuna.logging.set_verbosity(optuna.logging.WARNING)

RANDOM_STATE = 42
N_TRIALS = 50          # naikkan ke 100-200 kalau waktu memungkinkan

## 3. Upload Data

Upload 2 file: `*_training_features.npz` (berisi `X_morgan`, `X_pubchem`, `X_combined`, `y`)
dan `*_training_metadata.csv` (berisi `molecule_chembl_id`, `canonical_smiles`, `mean_standard_value`, `label`).

In [ ]:
from google.colab import files
uploaded = files.upload()  # pilih KEDUA file: .npz dan .csv sekaligus

npz_path = [f for f in uploaded.keys() if f.endswith(".npz")][0]
csv_path = [f for f in uploaded.keys() if f.endswith(".csv")][0]
print("NPZ  :", npz_path)
print("META :", csv_path)

Saving training_metadata.csv to training_metadata.csv
Saving training_features.npz to training_features.npz
NPZ  : training_features.npz
META : training_metadata.csv


## 4. Load Fitur + Scaffold Grouping

- Fitur diambil dari `X_combined` (Morgan FP + PubChem FP + RDKit descriptor yang sudah digabung).
  Ganti ke `X_morgan`, `X_pubchem`, atau gabungan manual kalau ingin eksperimen kombinasi fitur lain.
- `y` divalidasi harus sama persis dengan kolom `label` di metadata (urutan baris harus konsisten).
- Scaffold group dibuat dari `canonical_smiles` di metadata, dipakai untuk `GroupKFold`.

In [ ]:
def get_scaffold(smiles: str) -> str:
    """Ambil Murcko scaffold sebagai string SMILES, dipakai sebagai 'group id'."""
    try:
        mol = Chem.MolFromSmiles(smiles)
        scaffold = MurckoScaffold.GetScaffoldForMol(mol)
        return Chem.MolToSmiles(scaffold)
    except Exception:
        return smiles  # fallback kalau parsing gagal


def load_data(npz_path: str, csv_path: str, feature_key: str = "X_combined"):
    data = np.load(npz_path, allow_pickle=True)
    meta = pd.read_csv(csv_path)

    X = data[feature_key]
    y = data["y"]

    # Validasi konsistensi urutan baris antara npz dan metadata
    assert X.shape[0] == len(meta), "Jumlah baris fitur dan metadata tidak sama!"
    assert np.array_equal(y, meta["label"].values), "Label di npz dan csv tidak konsisten! Cek urutan baris."

    meta = meta.copy()
    meta["scaffold"] = meta["canonical_smiles"].apply(get_scaffold)
    meta["scaffold_group"] = meta["scaffold"].astype("category").cat.codes
    groups = meta["scaffold_group"].values

    return X, y, groups, meta


X, y, groups, meta = load_data(npz_path, csv_path, feature_key="X_morgan")
print(f"Jumlah data: {X.shape[0]}, Jumlah fitur: {X.shape[1]}")
print(f"Jumlah unique scaffold groups: {len(np.unique(groups))}")
print(f"Distribusi label:\n{pd.Series(y).value_counts()}")

Jumlah data: 2034, Jumlah fitur: 2265
Jumlah unique scaffold groups: 789
Distribusi label:
0    1329
1     705
Name: count, dtype: int64


## 5. Fungsi CV Berbasis Scaffold Group (dengan Pruning)

Perubahan untuk mempercepat proses:
- **CV 3-fold saat searching** (bukan 5-fold) — cukup untuk membandingkan kombinasi hyperparameter, lebih hemat waktu ~40%.
- **Optuna Pruning** — tiap fold, skor dilaporkan ke Optuna; kalau trial jelas lebih buruk dari trial-trial sebelumnya, langsung dihentikan (tidak perlu selesaikan semua fold).
- **Early stopping** untuk XGBoost/LightGBM/CatBoost — training berhenti otomatis begitu validation score berhenti membaik, tidak dipaksa sampai `n_estimators` penuh.


In [ ]:
N_SPLITS_SEARCH = 3   # CV saat hyperparameter search (lebih cepat)
N_SPLITS_FINAL = 5    # CV saat evaluasi final ensemble (lebih teliti)
EARLY_STOPPING_ROUNDS = 50


def cv_score_grouped_pruned(trial, model_builder, X, y, groups, n_splits=N_SPLITS_SEARCH,
                             use_early_stopping=False):
    """
    model_builder: fungsi yang menerima () -> model baru (unfitted)
    use_early_stopping: kalau True, model di-fit dengan eval_set + early stopping
                         (khusus XGB/LGB/CatBoost)
    """
    gkf = GroupKFold(n_splits=n_splits)
    scores = []

    for fold_idx, (train_idx, val_idx) in enumerate(gkf.split(X, y, groups)):
        X_tr, X_val = X[train_idx], X[val_idx]
        y_tr, y_val = y[train_idx], y[val_idx]
        model = model_builder()

        if use_early_stopping:
            # split kecil dari train untuk early stopping, val_idx tetap murni untuk skor
            n_holdout = max(int(0.15 * len(X_tr)), 50)
            X_fit, X_es = X_tr[:-n_holdout], X_tr[-n_holdout:]
            y_fit, y_es = y_tr[:-n_holdout], y_tr[-n_holdout:]
            model.fit(X_fit, y_fit, X_es, y_es)  # lihat wrapper fit() tiap model di bawah
        else:
            model.fit(X_tr, y_tr)

        proba = model.predict_proba(X_val)[:, 1]
        score = roc_auc_score(y_val, proba)
        scores.append(score)

        # laporkan skor rata-rata sejauh ini ke Optuna untuk pruning
        trial.report(np.mean(scores), step=fold_idx)
        if trial.should_prune():
            raise optuna.TrialPruned()

    return np.mean(scores)

## 6. Objective Functions untuk Optuna (per model, dengan early stopping)

Tiap model dibungkus dalam fungsi `fit(X, y, X_es=None, y_es=None)` yang seragam,
supaya `cv_score_grouped_pruned` bisa memanggilnya dengan cara yang sama.

In [ ]:
class RFWrapper:
    """RF tidak punya early stopping, fit() mengabaikan X_es/y_es."""
    def __init__(self, **params):
        self.model = RandomForestClassifier(**params, random_state=RANDOM_STATE, n_jobs=-1)

    def fit(self, X, y, X_es=None, y_es=None):
        self.model.fit(X, y)
        return self

    def predict_proba(self, X):
        return self.model.predict_proba(X)


class XGBWrapper:
    def __init__(self, **params):
        self.params = params

    def fit(self, X, y, X_es=None, y_es=None):
        self.model = XGBClassifier(
            **self.params, random_state=RANDOM_STATE, n_jobs=-1,
            eval_metric="logloss", early_stopping_rounds=EARLY_STOPPING_ROUNDS,
        )
        if X_es is not None:
            self.model.fit(X, y, eval_set=[(X_es, y_es)], verbose=False)
        else:
            self.model.fit(X, y)
        return self

    def predict_proba(self, X):
        return self.model.predict_proba(X)


class LGBWrapper:
    def __init__(self, **params):
        self.params = params

    def fit(self, X, y, X_es=None, y_es=None):
        import lightgbm as lgb
        self.model = LGBMClassifier(**self.params, random_state=RANDOM_STATE, n_jobs=-1, verbosity=-1)
        if X_es is not None:
            self.model.fit(
                X, y, eval_set=[(X_es, y_es)],
                callbacks=[lgb.early_stopping(EARLY_STOPPING_ROUNDS, verbose=False)],
            )
        else:
            self.model.fit(X, y)
        return self

    def predict_proba(self, X):
        return self.model.predict_proba(X)


class CatBoostWrapper:
    def __init__(self, **params):
        self.params = params

    def fit(self, X, y, X_es=None, y_es=None):
        self.model = CatBoostClassifier(
            **self.params, random_state=RANDOM_STATE, verbose=False, thread_count=-1,
        )
        if X_es is not None:
            self.model.fit(X, y, eval_set=(X_es, y_es), early_stopping_rounds=EARLY_STOPPING_ROUNDS)
        else:
            self.model.fit(X, y)
        return self

    def predict_proba(self, X):
        return self.model.predict_proba(X)


def objective_rf(trial, X, y, groups):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 500),
        "max_depth": trial.suggest_int("max_depth", 3, 25),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 10),
        "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", 0.3, 0.5]),
        "class_weight": trial.suggest_categorical("class_weight", ["balanced", None]),
    }
    return cv_score_grouped_pruned(trial, lambda: RFWrapper(**params), X, y, groups,
                                    use_early_stopping=False)


def objective_xgb(trial, X, y, groups):
    params = {
        "n_estimators": 2000,  # dibuat besar, biarkan early stopping yang menentukan titik henti
        "max_depth": trial.suggest_int("max_depth", 3, 12),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.3, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
    }
    return cv_score_grouped_pruned(trial, lambda: XGBWrapper(**params), X, y, groups,
                                    use_early_stopping=True)


def objective_lgb(trial, X, y, groups):
    params = {
        "n_estimators": 2000,
        "num_leaves": trial.suggest_int("num_leaves", 15, 200),
        "max_depth": trial.suggest_int("max_depth", 3, 15),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.3, 1.0),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 100),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
    }
    return cv_score_grouped_pruned(trial, lambda: LGBWrapper(**params), X, y, groups,
                                    use_early_stopping=True)


def objective_catboost(trial, X, y, groups):
    params = {
        "iterations": 2000,
        "depth": trial.suggest_int("depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1e-3, 10.0, log=True),
        "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 1.0),
        "random_strength": trial.suggest_float("random_strength", 1e-3, 10.0, log=True),
    }
    return cv_score_grouped_pruned(trial, lambda: CatBoostWrapper(**params), X, y, groups,
                                    use_early_stopping=True)

## 7. Jalankan Tuning untuk Semua Model (dengan Pruning)\n\nDengan CV 3-fold + pruning + early stopping, waktu tuning harusnya turun signifikan dibanding versi awal (bisa 3-5x lebih cepat).

In [ ]:
def tune_all_models(X, y, groups, n_trials=N_TRIALS):
    results = {}
    pruner = optuna.pruners.MedianPruner(n_warmup_steps=1)

    print("Tuning Random Forest...")
    study_rf = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE), pruner=pruner)
    study_rf.optimize(lambda t: objective_rf(t, X, y, groups), n_trials=n_trials, show_progress_bar=True)
    results["rf"] = study_rf.best_params
    print(f"  Best AUC: {study_rf.best_value:.4f}  |  Trials pruned: {sum(1 for t in study_rf.trials if t.state.name=='PRUNED')}/{len(study_rf.trials)}")

    print("Tuning XGBoost...")
    study_xgb = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE), pruner=pruner)
    study_xgb.optimize(lambda t: objective_xgb(t, X, y, groups), n_trials=n_trials, show_progress_bar=True)
    results["xgb"] = study_xgb.best_params
    print(f"  Best AUC: {study_xgb.best_value:.4f}  |  Trials pruned: {sum(1 for t in study_xgb.trials if t.state.name=='PRUNED')}/{len(study_xgb.trials)}")

    print("Tuning LightGBM...")
    study_lgb = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE), pruner=pruner)
    study_lgb.optimize(lambda t: objective_lgb(t, X, y, groups), n_trials=n_trials, show_progress_bar=True)
    results["lgb"] = study_lgb.best_params
    print(f"  Best AUC: {study_lgb.best_value:.4f}  |  Trials pruned: {sum(1 for t in study_lgb.trials if t.state.name=='PRUNED')}/{len(study_lgb.trials)}")

    print("Tuning CatBoost...")
    study_cat = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE), pruner=pruner)
    study_cat.optimize(lambda t: objective_catboost(t, X, y, groups), n_trials=n_trials, show_progress_bar=True)
    results["catboost"] = study_cat.best_params
    print(f"  Best AUC: {study_cat.best_value:.4f}  |  Trials pruned: {sum(1 for t in study_cat.trials if t.state.name=='PRUNED')}/{len(study_cat.trials)}")

    return results


best_params = tune_all_models(X, y, groups, n_trials=N_TRIALS)

print("\nBest hyperparameters:")
for model_name, params in best_params.items():
    print(f"  {model_name}: {params}")

Tuning Random Forest...


  0%|          | 0/50 [00:00<?, ?it/s]

  Best AUC: 0.8678  |  Trials pruned: 7/50
Tuning XGBoost...


  0%|          | 0/50 [00:00<?, ?it/s]

  Best AUC: 0.8215  |  Trials pruned: 4/50
Tuning LightGBM...


...

/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/ut

  Best AUC: 0.8113  |  Trials pruned: 2/50
Tuning CatBoost...


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


  0%|          | 0/50 [00:00<?, ?it/s]

  Best AUC: 0.8230  |  Trials pruned: 3/50

Best hyperparameters:
  rf: {'n_estimators': 137, 'max_depth': 19, 'min_samples_split': 3, 'min_samples_leaf': 1, 'max_features': 0.3, 'class_weight': None}
  xgb: {'max_depth': 11, 'learning_rate': 0.01392495115223496, 'subsample': 0.6266403849549695, 'colsample_bytree': 0.7233209987225901, 'min_child_weight': 2, 'reg_alpha': 0.35101202114883234, 'reg_lambda': 0.0010759929460773183}
  lgb: {'num_leaves': 99, 'max_depth': 13, 'learning_rate': 0.019721610970574007, 'subsample': 0.7571172192068059, 'colsample_bytree': 0.7146901982034297, 'min_child_samples': 9, 'reg_alpha': 0.0029369981104377003, 'reg_lambda': 3.425445902633376e-07}
  catboost: {'depth': 10, 'learning_rate': 0.017338551852422987, 'l2_leaf_reg': 0.2626226423317814, 'bagging_temperature': 0.277236517920161, 'random_strength': 0.01663369329384123}


## 7b. Cari Jumlah Estimator Optimal untuk Boosting (via Early Stopping)

Karena saat tuning XGB/LGB/CatBoost `n_estimators`/`iterations` di-set besar (2000) dan dihentikan
otomatis oleh early stopping, kita perlu jalankan sekali lagi dengan hyperparameter terbaik untuk
mendapatkan angka `n_estimators` final yang akan dipakai di model produksi (tanpa early stopping,
supaya bisa dipakai di dalam `StackingClassifier`).

In [ ]:
from sklearn.model_selection import train_test_split

def find_best_n_estimators(model_wrapper_cls, params, X, y, groups):
    # split sekali berdasarkan scaffold group untuk estimasi n_estimators
    gkf = GroupKFold(n_splits=5)
    train_idx, val_idx = next(gkf.split(X, y, groups))
    X_tr, X_val = X[train_idx], X[val_idx]
    y_tr, y_val = y[train_idx], y[val_idx]

    wrapper = model_wrapper_cls(**params)
    wrapper.fit(X_tr, y_tr, X_val, y_val)

    if hasattr(wrapper.model, "best_iteration"):        # XGBoost
        return wrapper.model.best_iteration + 1
    if hasattr(wrapper.model, "best_iteration_"):        # LightGBM
        return wrapper.model.best_iteration_
    if hasattr(wrapper.model, "get_best_iteration"):     # CatBoost
        return wrapper.model.get_best_iteration() + 1
    return params.get("n_estimators", 500)


best_n_xgb = find_best_n_estimators(XGBWrapper, {**best_params["xgb"], "n_estimators": 2000}, X, y, groups)
best_n_lgb = find_best_n_estimators(LGBWrapper, {**best_params["lgb"], "n_estimators": 2000}, X, y, groups)
best_n_cat = find_best_n_estimators(CatBoostWrapper, {**best_params["catboost"], "iterations": 2000}, X, y, groups)

best_params["xgb"]["n_estimators"] = best_n_xgb
best_params["lgb"]["n_estimators"] = best_n_lgb
best_params["catboost"]["iterations"] = best_n_cat

print(f"XGBoost n_estimators final: {best_n_xgb}")
print(f"LightGBM n_estimators final: {best_n_lgb}")
print(f"CatBoost iterations final: {best_n_cat}")

XGBoost n_estimators final: 188
LightGBM n_estimators final: 83
CatBoost iterations final: 72


## 8. Bangun 3 Ensemble: RF+XGB, RF+LGB, RF+CatBoost (Stacking)

In [ ]:
def build_ensembles(best_params):
    rf = RandomForestClassifier(**best_params["rf"], random_state=RANDOM_STATE, n_jobs=-1)
    xgb = XGBClassifier(**best_params["xgb"], random_state=RANDOM_STATE, n_jobs=-1, eval_metric="logloss")
    lgb = LGBMClassifier(**best_params["lgb"], random_state=RANDOM_STATE, n_jobs=-1, verbosity=-1)
    cat = CatBoostClassifier(**best_params["catboost"], random_state=RANDOM_STATE, verbose=False, thread_count=-1)

    meta_learner = LogisticRegression(max_iter=2000, class_weight="balanced")

    ensemble_rf_xgb = StackingClassifier(
        estimators=[("rf", rf), ("xgb", xgb)],
        final_estimator=meta_learner,
        cv=5,
        n_jobs=-1,
    )

    ensemble_rf_lgb = StackingClassifier(
        estimators=[("rf", rf), ("lgb", lgb)],
        final_estimator=meta_learner,
        cv=5,
        n_jobs=-1,
    )

    ensemble_rf_cat = StackingClassifier(
        estimators=[("rf", rf), ("catboost", cat)],
        final_estimator=meta_learner,
        cv=5,
    )

    return {
        "RF+XGB": ensemble_rf_xgb,
        "RF+LGB": ensemble_rf_lgb,
        "RF+CatBoost": ensemble_rf_cat,
    }


ensembles = build_ensembles(best_params)

## 9. Evaluasi dengan Scaffold-Aware GroupKFold

In [ ]:
def evaluate_ensembles(ensembles, X, y, groups, n_splits=N_SPLITS_FINAL):
    gkf = GroupKFold(n_splits=n_splits)
    report = {}

    for name, model in ensembles.items():
        aucs, f1s, accs = [], [], []
        for train_idx, val_idx in gkf.split(X, y, groups):
            X_tr, X_val = X[train_idx], X[val_idx]
            y_tr, y_val = y[train_idx], y[val_idx]
            model.fit(X_tr, y_tr)
            proba = model.predict_proba(X_val)[:, 1]
            pred = model.predict(X_val)
            aucs.append(roc_auc_score(y_val, proba))
            f1s.append(f1_score(y_val, pred))
            accs.append(accuracy_score(y_val, pred))

        report[name] = {
            "AUC_mean": np.mean(aucs), "AUC_std": np.std(aucs),
            "F1_mean": np.mean(f1s), "F1_std": np.std(f1s),
            "Acc_mean": np.mean(accs), "Acc_std": np.std(accs),
        }
        print(f"\n{name}:")
        print(f"  AUC = {report[name]['AUC_mean']:.4f} +/- {report[name]['AUC_std']:.4f}")
        print(f"  F1  = {report[name]['F1_mean']:.4f} +/- {report[name]['F1_std']:.4f}")
        print(f"  Acc = {report[name]['Acc_mean']:.4f} +/- {report[name]['Acc_std']:.4f}")

    return report


report = evaluate_ensembles(ensembles, X, y, groups)


RF+XGB:
  AUC = 0.8570 +/- 0.0393
  F1  = 0.7252 +/- 0.0490
  Acc = 0.8225 +/- 0.0232


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/ut


RF+LGB:
  AUC = 0.8538 +/- 0.0370
  F1  = 0.7142 +/- 0.0474
  Acc = 0.8156 +/- 0.0216

RF+CatBoost:
  AUC = 0.8624 +/- 0.0381
  F1  = 0.7327 +/- 0.0530
  Acc = 0.8235 +/- 0.0338


## 10. Pilih Ensemble Terbaik, Fit ke Seluruh Data, dan Simpan Model

In [ ]:
best_name = max(report, key=lambda k: report[k]["AUC_mean"])
print(f"Ensemble terbaik: {best_name}")

final_model = ensembles[best_name]
final_model.fit(X, y)

import joblib
model_filename = f"best_model_{best_name.replace('+', '_')}.pkl"
joblib.dump(final_model, model_filename)
print(f"Model tersimpan sebagai {model_filename}")

Ensemble terbaik: RF+CatBoost
Model tersimpan sebagai best_model_RF_CatBoost.pkl


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
PROJECT_DIR = '/content/drive/MyDrive/pla2'
DATA_DIR = os.path.join(PROJECT_DIR, 'data')
MODEL_DIR = os.path.join(PROJECT_DIR, 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

Mounted at /content/drive


In [ ]:
import joblib
import json

model_path = os.path.join(MODEL_DIR, 'final_model.joblib')
joblib.dump(final_model, model_path)

print('Model tersimpan di:', model_path)

Model tersimpan di: /content/drive/MyDrive/pla2/models/final_model.joblib


In [ ]:
# (Opsional) Download model hasil training ke komputer lokal
from google.colab import files
files.download(model_filename)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Catatan

- Kalau jumlah fitur fingerprint sangat besar (>2000 kolom), pertimbangkan **feature selection**
  (`VarianceThreshold` + filter korelasi tinggi) sebelum tuning untuk mempercepat proses.
- `N_TRIALS` bisa dinaikkan (100-200) kalau runtime Colab masih cukup & butuh hasil lebih optimal.
- Kalau data imbalance cukup parah, pertimbangkan metrik tambahan seperti **PR-AUC** atau **MCC**
  selain ROC-AUC.
